In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [6]:
acn = pd.read_excel(
    "acndata_sessions.json.xlsx"
)

print(acn.shape)

(16304, 27)


In [7]:
acn.isnull().sum()

_meta                 16304
end                   16304
min_kWh               16304
site                  16303
start                 16304
_items                16304
_id                    1305
clusterID              1305
connectionTime         1305
disconnectTime         1305
doneChargingTime       1313
kWhDelivered           1305
sessionID              1305
siteID                 1305
spaceID                1305
stationID              1305
timezone               1305
userID                14072
userInputs            16304
WhPerMile             12767
kWhRequested          12767
milesRequested        12767
minutesAvailable      12767
modifiedAt            12767
paymentRequired       12767
requestedDeparture    12767
userID.1              12767
dtype: int64

In [8]:
acn["kWhDelivered"].describe()

count    14999.000000
mean         9.002466
std          7.055848
min          0.501000
25%          4.008500
50%          7.435000
75%         13.204000
max         69.373000
Name: kWhDelivered, dtype: float64

In [9]:
acn = acn.dropna(
    subset=["kWhDelivered"]
)

print(acn.shape)

(14999, 27)


In [10]:
BASE_PRICE = 15

acn["baseline_revenue"] = (
    acn["kWhDelivered"]
    *
    BASE_PRICE
)

In [22]:
q25 = acn["kWhDelivered"].quantile(0.25)
q75 = acn["kWhDelivered"].quantile(0.75)

print(q25)
print(q75)

4.0085
13.204


In [23]:
def acn_tariff(kwh):

    if kwh >= q75:

        return 18

    elif kwh <= q25:

        return 14

    else:

        return 15

In [24]:
acn["dynamic_tariff"] = (
    acn["kWhDelivered"]
    .apply(acn_tariff)
)

In [25]:
acn["dynamic_tariff"].value_counts()

dynamic_tariff
15    7494
18    3755
14    3750
Name: count, dtype: int64

In [26]:
acn["dynamic_revenue"] = (

    acn["kWhDelivered"]

    *

    acn["dynamic_tariff"]

)

In [27]:
baseline_revenue = (
    acn["baseline_revenue"]
    .sum()
)

dynamic_revenue = (
    acn["dynamic_revenue"]
    .sum()
)

revenue_gain_pct_acn = (

    (
        dynamic_revenue
        -
        baseline_revenue
    )

    /

    baseline_revenue

) * 100

print(
    "Revenue Gain (%) =",
    revenue_gain_pct_acn
)

Revenue Gain (%) = 9.459682592051086


In [28]:
import numpy as np

acn["adjusted_energy"] = np.where(

    acn["dynamic_tariff"] == 14,

    acn["kWhDelivered"] * 1.20,

    np.where(

        acn["dynamic_tariff"] == 18,

        acn["kWhDelivered"] * 0.98,

        acn["kWhDelivered"]
    )
)

In [29]:
customer_response_rate = (

    (
        acn["adjusted_energy"].sum()

        -

        acn["kWhDelivered"].sum()
    )

    /

    acn["kWhDelivered"].sum()

) * 100

print(
    "Customer Response Rate (%) =",
    customer_response_rate
)

Customer Response Rate (%) = 0.22941313234030675


In [30]:
pricing_efficiency_score = (

    dynamic_revenue

    /

    acn["adjusted_energy"].sum()

)

print(
    "Pricing Efficiency Score =",
    pricing_efficiency_score
)

Pricing Efficiency Score = 16.38137137162372


In [31]:
acn_metrics = pd.DataFrame({

    "Metric":[

        "Revenue Gain (%)",

        "Customer Response Rate (%)",

        "Pricing Efficiency Score"
    ],

    "Value":[

        revenue_gain_pct_acn,

        customer_response_rate,

        pricing_efficiency_score
    ]
})

acn_metrics

,Metric,Value
0,Revenue Gain (%),9.459683
1,Customer Response Rate (%),0.229413
2,Pricing Efficiency Score,16.381371


In [34]:
acn_metrics.to_csv(
    "acn_metrics.csv",
    index=False
)

print("ACN metrics saved successfully")

ACN metrics saved successfully
